# F2-M01: Inspección del Dataset

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Inspección inicial del dataset: estructura, tipos de datos, cardinalidad,
valores únicos, porcentaje de nulos y uso de memoria.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m01_inspeccion.html`
- `docs/html/fase2/graficos/m01_tipos_datos.html`
- `docs/html/fase2/graficos/m01_nulos.html`
- `docs/html/fase2/graficos/m01_lollipop_cardinalidad.html`
- `docs/html/fase2/graficos/m01_dist_cardinalidad.html`
- `docs/html/fase2/graficos/m01_unicos.html`

## Flujo
```
M00 Índice → **M01 Inspección** → M02 Calidad → M03 Nulos → ...
```

## Siguiente
`f2_m02_calidad.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto y librerías de visualización
# - Crea directorios de salida para HTML y gráficos
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\FF\AU_UJI_v2
✓ 📁 RAW:           C:\FF\AU_UJI_v2\data\00_raw
✓ 📁 INTERIM:       C:\FF\AU_UJI_v2\data\01_interim
✓ 📁 PROCESSED:     C:\FF\AU_UJI_v2\data\02_processed
✓ 📁 FEATURES:      C:\FF\AU_UJI_v2\data\03_features
✓ 📁 AUTOML:        C:\FF\AU_UJI_v2\data\automl
✓ 📁 NOTEBOOKS:     C:\FF\AU_UJI_v2\notebooks
✓ 📄 Excel principal: C:\FF\AU_UJI_v2\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===========================================================================


In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CLASIFICAR TIPOS
# ============================================================================
# Lee df_alumno.parquet y clasifica columnas por tipo de dato.
# Calcula métricas globales: filas, columnas, alumnos únicos.
# ============================================================================

print('=' * 60)
print('INSPECCIÓN DEL DATASET')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# --- Métricas globales ---
n_filas = len(df)
n_cols = len(df.columns)
n_alumnos = df['per_id_ficticio'].nunique() if 'per_id_ficticio' in df.columns else n_filas

# --- Clasificar tipos de datos ---
n_numericas = len(df.select_dtypes(include=[np.number]).columns)
n_categoricas = len(df.select_dtypes(include=['object', 'category']).columns)
n_datetime = len(df.select_dtypes(include=['datetime64']).columns)
n_bool = len(df.select_dtypes(include=['bool']).columns)

print(f'✅ Dataset: {formato_numero_es(n_filas)} filas × {n_cols} columnas')
print(f'👥 Alumnos únicos: {formato_numero_es(n_alumnos)}')

INSPECCIÓN DEL DATASET
✅ Dataset: 109.568 filas × 37 columnas
👥 Alumnos únicos: 30.872


In [3]:
# ============================================================================
# CELDA 3: GRÁFICO 1 — DISTRIBUCIÓN DE TIPOS (DONUT)
# ============================================================================
# Gráfico circular con agujero central mostrando la proporción
# de cada tipo de dato en el dataset.
# ============================================================================

print('=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

tipos_conteo = df.dtypes.astype(str).value_counts()
colores_tipos = ['#3182ce', '#38a169', '#ed8936', '#e53e3e', '#805ad5']

fig_tipos = go.Figure(go.Pie(
    labels=tipos_conteo.index,
    values=tipos_conteo.values,
    hole=0.4,
    marker_colors=colores_tipos[:len(tipos_conteo)],
    textinfo='label+value+percent',
    textposition='outside'
))
fig_tipos.update_layout(
    title='📊 Distribución de Tipos de Datos',
    height=400,
    margin=dict(t=60, b=40, l=40, r=40)
)
fig_tipos.write_html(RUTA_GRAFICOS / 'm01_tipos_datos.html', include_plotlyjs='cdn')
print('✅ Gráfico tipos de datos')

GENERANDO GRÁFICOS


✅ Gráfico tipos de datos


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO 2 — % NULOS POR COLUMNA
# ============================================================================
# Barras horizontales con porcentaje de nulos.
# Solo muestra columnas que tienen algún nulo.
# Colores por severidad: verde (<5%), amarillo (5-20%),
# naranja (20-50%), rojo (>50%).
# ============================================================================

nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).sort_values(ascending=True)
nulos_pct = nulos_pct[nulos_pct > 0]

if len(nulos_pct) > 0:
    colores_nulos = [
        '#e53e3e' if v > 50 else '#ed8936' if v > 20 else '#ecc94b' if v > 5 else '#38a169'
        for v in nulos_pct.values
    ]

    fig_nulos = go.Figure(go.Bar(
        x=nulos_pct.values,
        y=nulos_pct.index,
        orientation='h',
        marker_color=colores_nulos,
        text=[f'{v:.1f}%' for v in nulos_pct.values],
        textposition='outside'
    ))
    fig_nulos.update_layout(
        title='❓ % Nulos por Columna<br><span style="font-size:11px; color:#718096;">🟢 <5% | 🟡 5-20% | 🟠 20-50% | 🔴 >50%</span>',
        xaxis_title='% Nulos',
        height=400,
        margin=dict(t=80, b=50, l=180, r=80)
    )
else:
    fig_nulos = go.Figure()
    fig_nulos.add_annotation(text='✅ Sin valores nulos', x=0.5, y=0.5, showarrow=False, font_size=20)
    fig_nulos.update_layout(title='❓ % Nulos por Columna<br><span style="font-size:11px; color:#718096;">🟢 <5% | 🟡 5-20% | 🟠 20-50% | 🔴 >50%</span>', height=400)

fig_nulos.write_html(RUTA_GRAFICOS / 'm01_nulos.html', include_plotlyjs='cdn')
print('✅ Gráfico nulos')

✅ Gráfico nulos


In [5]:
# ============================================================================
# CELDA 5: GRÁFICO 3 — LOLLIPOP DE CARDINALIDAD (nunique)
# ============================================================================
# Lollipop chart horizontal: línea fina + punto para cada columna.
# Muestra nunique() por columna, ordenado de mayor a menor.
# Escala logarítmica para ver bien IDs (miles) y binarias (2).
# Colores por categoría:
#   🔴 Rojo = posible ID/clave (>50% del total de filas)
#   🟢 Verde = cardinalidad normal
#   🟣 Morado = baja cardinalidad (<10 únicos)
# ============================================================================

unicos_por_col = df.nunique().sort_values(ascending=True)

# Clasificar por categoría de cardinalidad
# Mismos rangos y colores que el gráfico de distribución de cardinalidades
colores_lollipop = []
for col_name in unicos_por_col.index:
    n_u = unicos_por_col[col_name]
    if n_u <= 2:
        colores_lollipop.append('#805ad5')   # Binaria
    elif n_u <= 10:
        colores_lollipop.append('#9AE6B4')   # Baja
    elif n_u <= 100:
        colores_lollipop.append('#38a169')   # Media
    elif n_u <= 1000:
        colores_lollipop.append('#ed8936')   # Alta
    else:
        colores_lollipop.append('#e53e3e')   # Muy alta

fig_lollipop = go.Figure()

# Líneas del lollipop (de 0 al valor)
for i, (col_name, val) in enumerate(unicos_por_col.items()):
    fig_lollipop.add_trace(go.Scatter(
        x=[0.5, val],  # 0.5 para que se vea con escala log
        y=[col_name, col_name],
        mode='lines',
        line=dict(color=colores_lollipop[i], width=2),
        showlegend=False,
        hoverinfo='skip'
    ))

# Puntos del lollipop con etiquetas
fig_lollipop.add_trace(go.Scatter(
    x=unicos_por_col.values,
    y=unicos_por_col.index,
    mode='markers+text',
    marker=dict(
        size=10,
        color=colores_lollipop,
        line=dict(width=1, color='white')
    ),
    text=[f'{v:,}' for v in unicos_por_col.values],
    textposition='middle right',
    textfont=dict(size=9),
    showlegend=False,
    hovertemplate='%{y}: %{x:,} valores únicos<extra></extra>'
))

fig_lollipop.update_layout(
    title='🎯 Cardinalidad por Columna (valores únicos)<br><span style="font-size:11px; color:#718096;">🟣 Binaria (≤2) | 🟢 Baja (3-10) | 🟢 Media (11-100) | 🟠 Alta (101-1K) | 🔴 Muy alta (>1K)</span>',
    xaxis_title='Nº Valores Únicos (escala log)',
    xaxis_type='log',
    height=max(500, len(unicos_por_col) * 22),
    margin=dict(t=80, b=80, l=180, r=100)
)

fig_lollipop.write_html(RUTA_GRAFICOS / 'm01_lollipop_cardinalidad.html', include_plotlyjs='cdn')
print('✅ Lollipop cardinalidad')

✅ Lollipop cardinalidad


In [6]:
# ============================================================================
# CELDA 6: GRÁFICO 4 — DISTRIBUCIÓN DE CARDINALIDADES (RANGOS)
# ============================================================================
# Cuántas columnas caen en cada rango de cardinalidad.
# Da una visión panorámica de la complejidad del dataset.
# Rangos: Binaria (≤2), Baja (3-10), Media (11-100),
#         Alta (101-1K), Muy alta (>1K)
# ============================================================================

rangos = []
for col_name in df.columns:
    n_u = df[col_name].nunique()
    if n_u <= 2:
        rangos.append('Binaria (≤2)')
    elif n_u <= 10:
        rangos.append('Baja (3-10)')
    elif n_u <= 100:
        rangos.append('Media (11-100)')
    elif n_u <= 1000:
        rangos.append('Alta (101-1K)')
    else:
        rangos.append('Muy alta (>1K)')

# Contar columnas por rango (mantener orden lógico)
orden_rangos = ['Binaria (≤2)', 'Baja (3-10)', 'Media (11-100)', 'Alta (101-1K)', 'Muy alta (>1K)']
conteo_rangos = pd.Series(rangos).value_counts().reindex(orden_rangos, fill_value=0)

colores_rangos = ['#805ad5', '#9AE6B4', '#38a169', '#ed8936', '#e53e3e']

fig_dist_card = go.Figure(go.Bar(
    x=conteo_rangos.index,
    y=conteo_rangos.values,
    marker_color=colores_rangos,
    text=conteo_rangos.values,
    textposition='outside',
    textfont=dict(size=14, color='#2d3748')
))

fig_dist_card.update_layout(
    title='📊 Distribución de Cardinalidades<br><span style="font-size:11px; color:#718096;">Misma leyenda de colores que el gráfico izquierdo</span>',
    xaxis_title='Rango de Cardinalidad',
    yaxis_title='Nº de Columnas',
    height=400,
    margin=dict(t=80, b=60, l=60, r=40),
    yaxis=dict(dtick=1)
)

fig_dist_card.write_html(RUTA_GRAFICOS / 'm01_dist_cardinalidad.html', include_plotlyjs='cdn')
print('✅ Distribución de cardinalidades')

✅ Distribución de cardinalidades


In [7]:
# ============================================================================
# CELDA 7: GRÁFICO 5 — VALORES ÚNICOS POR COLUMNA (TOP 20)
# ============================================================================
# Barras horizontales mostrando las 20 columnas con más
# valores distintos. Útil para detectar posibles IDs.
# ============================================================================

unicos = df.nunique().sort_values(ascending=True).tail(20)

fig_unicos = go.Figure(go.Bar(
    x=unicos.values,
    y=unicos.index,
    orientation='h',
    marker_color='#805ad5',
    text=[f'{v:,}' for v in unicos.values],
    textposition='outside'
))
fig_unicos.update_layout(
    title='🔢 Valores Únicos por Columna (Top 20)',
    xaxis_title='Valores Únicos',
    height=450,
    margin=dict(t=60, b=50, l=180, r=80)
)
fig_unicos.write_html(RUTA_GRAFICOS / 'm01_unicos.html', include_plotlyjs='cdn')
print('✅ Gráfico valores únicos')

✅ Gráfico valores únicos


In [8]:
# ============================================================================
# CELDA 8: TABLA DETALLADA DE COLUMNAS
# ============================================================================
# DataFrame resumen: columna, dtype, tipo clasificado,
# valores válidos, nulos, % nulos y únicos.
# ============================================================================

print('\n📋 Preparando tabla detallada...')

resumen_cols = []
for col in df.columns:
    dtype = str(df[col].dtype)
    n_validos = df[col].notna().sum()
    n_nulos = df[col].isnull().sum()
    pct_nulos = (n_nulos / len(df) * 100)
    n_unicos = df[col].nunique()

    # Clasificar tipo semántico
    if 'int' in dtype or 'float' in dtype:
        tipo = 'Numérica'
    elif 'datetime' in dtype:
        tipo = 'Datetime'
    elif 'bool' in dtype:
        tipo = 'Booleana'
    else:
        tipo = 'Categórica'

    resumen_cols.append({
        'Columna': col,
        'Dtype': dtype,
        'Tipo': tipo,
        'Válidos': n_validos,
        'Nulos': n_nulos,
        '%Nulos': pct_nulos,
        'Únicos': n_unicos
    })

df_resumen = pd.DataFrame(resumen_cols)
print(f'✅ Tabla preparada: {len(df_resumen)} columnas')


📋 Preparando tabla detallada...


✅ Tabla preparada: 37 columnas


In [9]:
# ============================================================================
# CELDA: GENERAR HTML
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- KPIs ---
KPIS = [
    {'valor': str(len(df.columns)), 'titulo': 'Variables'},
    {'valor': formato_numero_es(len(df)), 'titulo': 'Registros'},
    {'valor': str(df.select_dtypes(include='number').shape[1]), 'titulo': 'Numéricas'},
    {'valor': str(df.select_dtypes(exclude='number').shape[1]), 'titulo': 'No Numéricas'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Tabla resumen ---
tabla_html = df_resumen.to_html(classes='tabla-estadisticas', index=False,
    float_format=lambda x: f'{x:.1f}')

# --- Secciones con iframes ---
altura_lollipop = max(400, len(df.columns) * 18)
seccion_tipos = generar_seccion_html(
    titulo='Tipos de Datos', icono='📊',
    contenido='''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m01_tipos_datos.html" width="100%" height="420" frameborder="0"></iframe>
        <iframe src="graficos/m01_nulos.html" width="100%" height="420" frameborder="0"></iframe>
    </div>'''
)
seccion_card = generar_seccion_html(
    titulo='Cardinalidad y Valores Únicos', icono='🔢',
    contenido=f'''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m01_lollipop_cardinalidad.html" width="100%" height="{altura_lollipop}" frameborder="0"></iframe>
        <iframe src="graficos/m01_dist_cardinalidad.html" width="100%" height="420" frameborder="0"></iframe>
    </div>'''
)
seccion_unicos = generar_seccion_html(
    titulo='Top 20 Columnas por Valores Únicos', icono='🔝',
    contenido='<iframe src="graficos/m01_unicos.html" width="100%" height="470" frameborder="0"></iframe>'
)
seccion_tabla = generar_seccion_html(
    titulo='Detalle de Columnas', icono='📋',
    contenido=f'<div style="overflow-x:auto; max-height:400px; overflow-y:auto;">{tabla_html}</div>'
)

contenido_html = kpis_html + seccion_tipos + seccion_card + seccion_unicos + seccion_tabla

# --- Generar HTML ---
html_completo = render_pagina_desde_fichero(
    'f2_m01_inspeccion.ipynb',
    contenido_html,
    carpeta_notebook='fase2_eda'
)
ruta_html = RUTA_FASE2 / "m01_inspeccion.html"
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')
print(f"\n✅ HTML: {ruta_html}")


GENERANDO HTML


✅ HTML generado: C:\FF\AU_UJI_v2\docs\html\fase2\m01_inspeccion.html

✅ HTML: C:\FF\AU_UJI_v2\docs\html\fase2\m01_inspeccion.html


In [10]:
# ============================================================================
# CELDA 10: RESUMEN DE EJECUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M01 COMPLETADO')
print('=' * 60)
print('📊 Fila 1: Tipos (donut) + % Nulos')
print('🎯 Fila 2: Lollipop cardinalidad + Distribución rangos')
print('🔢 Fila 3: Top 20 valores únicos')
print('📋 Tabla: Detalle de columnas')
print('📌 Siguiente: f2_m02_calidad.ipynb')


✅ F2-M01 COMPLETADO
📊 Fila 1: Tipos (donut) + % Nulos
🎯 Fila 2: Lollipop cardinalidad + Distribución rangos
🔢 Fila 3: Top 20 valores únicos
📋 Tabla: Detalle de columnas
📌 Siguiente: f2_m02_calidad.ipynb
